# V18-C11: LIANA Tissue-Separated Cell-Cell Interaction Analysis
## Simpson's Paradox-Aware: 2-Track (Discovery + Donor-Level Validation)

**Date:** 2026-03-05
**Purpose:**
1. Verify V18 paracrine interaction hypotheses
2. Compare Liver vs Blood interaction landscapes
3. Compare with Zhang et al. CSOmap results

**Critical Design:**
- **Track A (Discovery):** Pooled LIANA per tissue×disease — exploratory, pseudo-replication present
- **Track B (Validation):** Donor-level ligand-receptor co-expression → Mann-Whitney U — consistent with V18 statistical framework
- Track A findings MUST be confirmed by Track B to be considered robust

**Statistical unit:** Donor-level (Track B). Cell-level LIANA (Track A) = discovery only.

In [1]:
# ============================================================
# CELL 0: GPU SETUP & VERIFICATION
# ============================================================
!pip install scanpy anndata matplotlib seaborn scipy -q

import subprocess
import sys

print("=" * 70)
print("  V18 C3: GPU ENVIRONMENT SETUP")
print("=" * 70)

# GPU detection
try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,compute_cap',
                            '--format=csv,noheader'], capture_output=True, text=True)
    gpu_info = result.stdout.strip()
    print(f"✅ GPU detected: {gpu_info}")
except:
    print("⚠️ No GPU detected — will use CPU fallback")

# Install CuPy for GPU acceleration
try:
    import cupy as cp
    print(f"✅ CuPy {cp.__version__} ready")
    print(f"   GPU memory: {cp.cuda.Device(0).mem_info[1] / 1e9:.1f} GB total")
except ImportError:
    print("📦 Installing CuPy...")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'cupy-cuda12x', '-q'])
    import cupy as cp
    print(f"✅ CuPy {cp.__version__} installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.1/284.1 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 140.1 MB/s eta 0:00:00
  V18 C3: GPU ENVIRONMENT SETUP
✅ GPU detected: NVIDIA A100-SXM4-40GB, 40960 MiB, 8.0
✅ CuPy 14.0.1 ready
   GPU memory: 42.4 GB total


---
## Cell 1: Install & Setup

In [2]:
# ============================================================
# Cell 1: Install and setup
# ============================================================
!pip install liana --quiet

import liana
print(f'LIANA version: {liana.__version__}')

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib
import warnings
from scipy.stats import mannwhitneyu, spearmanr
from scipy import sparse
warnings.filterwarnings('ignore')
matplotlib.rcParams['figure.dpi'] = 150
matplotlib.rcParams['figure.facecolor'] = 'white'

from google.colab import drive
drive.mount('/content/drive')

import os
RESULTS_DIR = '/content/drive/MyDrive/ITLAS/results/version18-analysis/C11_LIANA/'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Setup complete.')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43.4 kB 5.2 MB/s eta 0:00:00
LIANA version: 1.7.1
Mounted at /content/drive
Setup complete.


---
## Cell 2: Load Data (C10 conventions)

In [3]:
# ============================================================
# Cell 2: Load data
# ============================================================
DATA_PATH = '/content/drive/MyDrive/ITLAS/data/processed/GSE182159_gut2021_annotated.h5ad'
RESULTS_DIR = '/content/drive/MyDrive/ITLAS/results/version18-analysis/C11B_LIANA_DonorValidated/'
print('Loading h5ad...')
adata = sc.read_h5ad(DATA_PATH)
print(f'Loaded: {adata.shape[0]} cells x {adata.shape[1]} genes')

COL_DISEASE = 'Stage'
COL_LINEAGE = 'major_lineage'
COL_TISSUE = 'tissue'
COL_SUBCLUSTER = 'gut2021_subcluster_v2'
adata.obs['donor_id'] = adata.obs['sample'].str.split('_').str[1]
COL_DONOR = 'donor_id'

TISSUES = ['Liver', 'Blood']
DISEASE_GROUPS = ['NL', 'IT', 'IA', 'AR', 'CR']
LINEAGES = ['Myeloid', 'CD4_T', 'CD8_T', 'NK', 'B', 'PlasmaB']

print(f'Donors: {adata.obs[COL_DONOR].nunique()}')
print(f'Ready.')

Loading h5ad...
Loaded: 243000 cells x 24452 genes
Donors: 23
Ready.


---
## Cell 3: Define V18 Hypothesized Interactions

In [4]:
# ============================================================
# Cell 3: V18 hypothesized L-R pairs to verify
# ============================================================
V18_LR_PAIRS = [
    # Pattern 1: Myeloid paracrine
    {'name': 'TGFB1_TGFBR2', 'ligand': 'TGFB1', 'receptor': 'TGFBR2',
     'source_lineage': 'Myeloid', 'target_lineages': ['CD4_T', 'CD8_T', 'NK', 'B']},
    {'name': 'TGFB1_TGFBR1', 'ligand': 'TGFB1', 'receptor': 'TGFBR1',
     'source_lineage': 'Myeloid', 'target_lineages': ['CD4_T', 'CD8_T', 'NK']},
    {'name': 'LGALS9_HAVCR2', 'ligand': 'LGALS9', 'receptor': 'HAVCR2',
     'source_lineage': 'Myeloid', 'target_lineages': ['CD4_T', 'CD8_T', 'NK']},
    # HLA-mediated Ag presentation
    {'name': 'HLA-DRA_CD4', 'ligand': 'HLA-DRA', 'receptor': 'CD4',
     'source_lineage': 'Myeloid', 'target_lineages': ['CD4_T']},
    # Additional suppressive
    {'name': 'LGALS9_CD44', 'ligand': 'LGALS9', 'receptor': 'CD44',
     'source_lineage': 'Myeloid', 'target_lineages': ['CD4_T', 'CD8_T']},
    # B cell activation
    {'name': 'IL2_IL2RA', 'ligand': 'IL2', 'receptor': 'IL2RA',
     'source_lineage': 'CD4_T', 'target_lineages': ['B']},
    # NK exhaustion
    {'name': 'TGFB1_TGFBR2_NK', 'ligand': 'TGFB1', 'receptor': 'TGFBR2',
     'source_lineage': 'NK', 'target_lineages': ['CD4_T', 'CD8_T']},
]

print(f'Defined {len(V18_LR_PAIRS)} L-R pairs to verify')
for pair in V18_LR_PAIRS:
    print(f"  {pair['name']}: {pair['source_lineage']} ({pair['ligand']}) -> "
          f"{pair['target_lineages']} ({pair['receptor']})")

Defined 7 L-R pairs to verify
  TGFB1_TGFBR2: Myeloid (TGFB1) -> ['CD4_T', 'CD8_T', 'NK', 'B'] (TGFBR2)
  TGFB1_TGFBR1: Myeloid (TGFB1) -> ['CD4_T', 'CD8_T', 'NK'] (TGFBR1)
  LGALS9_HAVCR2: Myeloid (LGALS9) -> ['CD4_T', 'CD8_T', 'NK'] (HAVCR2)
  HLA-DRA_CD4: Myeloid (HLA-DRA) -> ['CD4_T'] (CD4)
  LGALS9_CD44: Myeloid (LGALS9) -> ['CD4_T', 'CD8_T'] (CD44)
  IL2_IL2RA: CD4_T (IL2) -> ['B'] (IL2RA)
  TGFB1_TGFBR2_NK: NK (TGFB1) -> ['CD4_T', 'CD8_T'] (TGFBR2)


---
## TRACK A: Pooled LIANA (Discovery)
### Cell 4: Run LIANA per tissue × disease (pooled across donors)
⚠️ **Pseudo-replication present.** Results are exploratory only.

In [5]:
# ============================================================
# Cell 4: Track A — Pooled LIANA (discovery)
# ⚠️ Cell-level analysis = pseudo-replication
# Use ONLY for discovery, NOT for statistical claims
# ============================================================

liana_pooled = {}

for tissue in TISSUES:
    for disease in DISEASE_GROUPS:
        key = f'{tissue}_{disease}'
        print(f'\n=== Track A (Pooled): {key} ===')

        mask = (adata.obs[COL_TISSUE] == tissue) & (adata.obs[COL_DISEASE] == disease)
        sub = adata[mask].copy()

        if sub.shape[0] < 100:
            print(f'  SKIP: {sub.shape[0]} cells')
            continue

        # Filter lineages with >= 10 cells
        lc = sub.obs[COL_LINEAGE].value_counts()
        valid = lc[lc >= 10].index.tolist()
        sub = sub[sub.obs[COL_LINEAGE].isin(valid)].copy()

        print(f'  Cells: {sub.shape[0]}, Lineages: {valid}')

        try:
            liana.mt.rank_aggregate(
                sub,
                groupby=COL_LINEAGE,
                resource_name='consensus',
                expr_prop=0.1,
                verbose=False,
                use_raw=False,
            )
            result = sub.uns['liana_res'].copy()
            result['tissue'] = tissue
            result['disease'] = disease
            liana_pooled[key] = result
            print(f'  ✅ {len(result)} interactions (⚠️ pooled, not donor-validated)')
        except Exception as e:
            print(f'  ❌ ERROR: {e}')

print(f'\n==> Track A complete: {len(liana_pooled)} conditions')


=== Track A (Pooled): Liver_NL ===
  Cells: 24219, Lineages: ['NK', 'CD8_T', 'CD4_T', 'B', 'PlasmaB', 'Myeloid', 'gdT']
  ✅ 4018 interactions (⚠️ pooled, not donor-validated)

=== Track A (Pooled): Liver_IT ===
  Cells: 19501, Lineages: ['CD8_T', 'NK', 'CD4_T', 'Myeloid', 'B', 'PlasmaB', 'gdT']
  ✅ 6111 interactions (⚠️ pooled, not donor-validated)

=== Track A (Pooled): Liver_IA ===
  Cells: 32289, Lineages: ['CD8_T', 'CD4_T', 'NK', 'Myeloid', 'B', 'PlasmaB', 'gdT']
  ✅ 4236 interactions (⚠️ pooled, not donor-validated)

=== Track A (Pooled): Liver_AR ===
  Cells: 17746, Lineages: ['CD8_T', 'CD4_T', 'NK', 'B', 'Myeloid', 'gdT', 'PlasmaB']
  ✅ 4756 interactions (⚠️ pooled, not donor-validated)

=== Track A (Pooled): Liver_CR ===
  Cells: 12837, Lineages: ['CD8_T', 'NK', 'CD4_T', 'B', 'Myeloid', 'PlasmaB', 'gdT']
  ✅ 5100 interactions (⚠️ pooled, not donor-validated)

=== Track A (Pooled): Blood_NL ===
  Cells: 18360, Lineages: ['NK', 'CD4_T', 'CD8_T', 'Myeloid', 'B', 'PlasmaB', 'gdT']

---
## TRACK B: Donor-Level Interaction Validation
### Cell 5: Compute donor-level L-R co-expression scores
**This is the V18-consistent statistical approach.**

For each donor × tissue × L-R pair:
- Compute mean ligand expression in source lineage
- Compute mean receptor expression in target lineage
- Interaction score = ligand_mean × receptor_mean (geometric mean proxy)
- Compare NL vs IT using Mann-Whitney U (donor-level, n=5~6)

In [7]:
# ============================================================
# Cell 5: Track B — Donor-level L-R co-expression
# This is the REAL statistical test
# ============================================================

import os # Import os module if not already imported in this scope

def safe_get_expr(adata_sub, gene):
    """Sparse/Dense safe gene expression extraction."""
    if gene not in adata_sub.var_names:
        return None
    gene_idx = list(adata_sub.var_names).index(gene)
    X = adata_sub.X
    if sparse.issparse(X):
        return np.asarray(X[:, gene_idx].todense()).flatten()
    else:
        return np.asarray(X[:, gene_idx]).flatten()

def compute_donor_lr_scores(adata, tissue, ligand, receptor,
                            source_lineage, target_lineage):
    """
    For each donor in a tissue:
      - mean ligand expression in source lineage
      - mean receptor expression in target lineage
      - interaction score = ligand_mean * receptor_mean
    Returns dict: {donor_id: {'ligand_mean', 'receptor_mean', 'interaction_score', 'disease'}}
    """
    tissue_mask = adata.obs[COL_TISSUE] == tissue
    tissue_data = adata[tissue_mask]

    results = []
    for donor in tissue_data.obs[COL_DONOR].unique():
        donor_mask = tissue_data.obs[COL_DONOR] == donor
        disease = tissue_data.obs.loc[donor_mask, COL_DISEASE].iloc[0]

        # Source lineage: ligand expression
        src_mask = donor_mask & (tissue_data.obs[COL_LINEAGE] == source_lineage)
        src_cells = tissue_data[src_mask]
        if src_cells.shape[0] < 3:  # minimum cells
            continue
        lig_expr = safe_get_expr(src_cells, ligand)
        if lig_expr is None:
            continue
        lig_mean = np.mean(lig_expr)

        # Target lineage: receptor expression
        tgt_mask = donor_mask & (tissue_data.obs[COL_LINEAGE] == target_lineage)
        tgt_cells = tissue_data[tgt_mask]
        if tgt_cells.shape[0] < 3:
            continue
        rec_expr = safe_get_expr(tgt_cells, receptor)
        if rec_expr is None:
            continue
        rec_mean = np.mean(rec_expr)

        # Interaction score: product of means
        interaction_score = lig_mean * rec_mean

        results.append({
            'donor': donor,
            'disease': disease,
            'tissue': tissue,
            'ligand': ligand,
            'receptor': receptor,
            'source_lineage': source_lineage,
            'target_lineage': target_lineage,
            'ligand_mean': lig_mean,
            'receptor_mean': rec_mean,
            'interaction_score': interaction_score,
            'n_source_cells': src_cells.shape[0],
            'n_target_cells': tgt_cells.shape[0],
        })

    return pd.DataFrame(results)

# Run for all V18 hypothesized L-R pairs
print('=' * 80)
print('  TRACK B: DONOR-LEVEL L-R INTERACTION SCORES')
print('=' * 80)

all_donor_lr = []

for pair in V18_LR_PAIRS:
    for target_lin in pair['target_lineages']:
        for tissue in TISSUES:
            df = compute_donor_lr_scores(
                adata, tissue,
                pair['ligand'], pair['receptor'],
                pair['source_lineage'], target_lin
            )
            if len(df) > 0:
                df['pair_name'] = pair['name']
                all_donor_lr.append(df)

donor_lr_df = pd.concat(all_donor_lr, ignore_index=True)
print(f'\nTotal donor-level L-R scores: {len(donor_lr_df)}')
print(f'Unique pairs: {donor_lr_df["pair_name"].nunique()}')

# Ensure the results directory exists before saving
os.makedirs(RESULTS_DIR, exist_ok=True)
donor_lr_df.to_csv(f'{RESULTS_DIR}C11_donor_level_LR_scores.csv', index=False)
print('Saved.')

  TRACK B: DONOR-LEVEL L-R INTERACTION SCORES

Total donor-level L-R scores: 673
Unique pairs: 7
Saved.


### Cell 6: Donor-Level Mann-Whitney U for L-R Interaction Scores
**This is the definitive statistical test — consistent with V18 framework.**

In [8]:
# ============================================================
# Cell 6: Track B — Mann-Whitney U on donor-level interaction scores
# NL vs IT, NL vs IA, IT vs IA, IA vs AR
# ============================================================

comparisons = [('NL', 'IT'), ('NL', 'IA'), ('IT', 'IA'), ('NL', 'CR'), ('IA', 'AR')]
lr_test_results = []

for g1, g2 in comparisons:
    print(f'\n=== {g1} vs {g2}: Donor-Level L-R Interaction Tests ===')

    for tissue in TISSUES:
        t_df = donor_lr_df[donor_lr_df['tissue'] == tissue]

        # Get unique pair × target lineage combinations
        combos = t_df.groupby(['pair_name', 'source_lineage', 'target_lineage',
                               'ligand', 'receptor']).size().reset_index()[:-1]

        for _, combo in t_df.groupby(['pair_name', 'source_lineage', 'target_lineage',
                                       'ligand', 'receptor']):
            pair_name = combo['pair_name'].iloc[0]
            src = combo['source_lineage'].iloc[0]
            tgt = combo['target_lineage'].iloc[0]
            lig = combo['ligand'].iloc[0]
            rec = combo['receptor'].iloc[0]

            vals_g1 = combo[combo['disease'] == g1]['interaction_score'].values
            vals_g2 = combo[combo['disease'] == g2]['interaction_score'].values

            if len(vals_g1) < 2 or len(vals_g2) < 2:
                continue

            mean_g1 = np.mean(vals_g1)
            mean_g2 = np.mean(vals_g2)

            try:
                _, pval = mannwhitneyu(vals_g1, vals_g2, alternative='two-sided')
            except:
                pval = 1.0

            if mean_g1 > 1e-10:
                pct_change = ((mean_g2 - mean_g1) / mean_g1) * 100
            else:
                pct_change = 99999 if mean_g2 > 1e-10 else 0

            sig = '★' if pval < 0.05 else ''

            lr_test_results.append({
                'comparison': f'{g1}→{g2}',
                'tissue': tissue,
                'pair_name': pair_name,
                'source': src, 'target': tgt,
                'ligand': lig, 'receptor': rec,
                'mean_g1': mean_g1, 'mean_g2': mean_g2,
                'pct_change': pct_change,
                'p_value': pval,
                'n_g1': len(vals_g1), 'n_g2': len(vals_g2),
                'significant': pval < 0.05
            })

            if pval < 0.1:  # show marginally significant
                print(f'  {sig}{tissue} {src}({lig})→{tgt}({rec}): '
                      f'{g1}={mean_g1:.4f} {g2}={mean_g2:.4f} '
                      f'{pct_change:+.1f}% p={pval:.4f}')

lr_test_df = pd.DataFrame(lr_test_results)
lr_test_df.to_csv(f'{RESULTS_DIR}C11_donor_level_LR_tests.csv', index=False)

# Summary
print(f'\n=== SUMMARY ===')
for comp in ['NL→IT', 'NL→IA', 'IT→IA', 'NL→CR', 'IA→AR']:
    comp_df = lr_test_df[lr_test_df['comparison'] == comp]
    sig = comp_df[comp_df['significant']]
    print(f'  {comp}: {len(sig)}/{len(comp_df)} significant interactions (p<0.05)')


=== NL vs IT: Donor-Level L-R Interaction Tests ===
  ★Liver Myeloid(HLA-DRA)→CD4_T(CD4): NL=0.8550 IT=1.8609 +117.6% p=0.0260
  Liver Myeloid(LGALS9)→CD4_T(CD44): NL=0.8980 IT=1.3561 +51.0% p=0.0931
  Liver Myeloid(LGALS9)→CD8_T(CD44): NL=0.8202 IT=1.2341 +50.5% p=0.0649
  Liver Myeloid(LGALS9)→NK(HAVCR2): NL=0.2243 IT=0.4062 +81.1% p=0.0931
  ★Blood Myeloid(HLA-DRA)→CD4_T(CD4): NL=0.9014 IT=1.7064 +89.3% p=0.0159
  ★Blood Myeloid(LGALS9)→CD4_T(CD44): NL=0.7321 IT=1.5727 +114.8% p=0.0159
  ★Blood Myeloid(LGALS9)→CD8_T(CD44): NL=0.6338 IT=1.3174 +107.9% p=0.0159
  ★Blood Myeloid(TGFB1)→CD4_T(TGFBR1): NL=0.0302 IT=0.1125 +272.1% p=0.0159
  ★Blood Myeloid(TGFB1)→CD8_T(TGFBR1): NL=0.0372 IT=0.1827 +391.2% p=0.0159
  ★Blood Myeloid(TGFB1)→NK(TGFBR1): NL=0.0903 IT=0.3948 +337.4% p=0.0159
  ★Blood Myeloid(TGFB1)→B(TGFBR2): NL=0.1996 IT=0.7401 +270.8% p=0.0159
  ★Blood Myeloid(TGFB1)→CD4_T(TGFBR2): NL=0.2086 IT=0.7611 +264.8% p=0.0159
  ★Blood Myeloid(TGFB1)→CD8_T(TGFBR2): NL=0.2184 IT=0.915

---
## Cell 7: Cross-Validate Track A vs Track B

In [9]:
# ============================================================
# Cell 7: Cross-validate pooled LIANA (Track A) vs donor-level (Track B)
# Question: Do Track A discoveries survive Track B validation?
# ============================================================

print('=' * 80)
print('  TRACK A vs TRACK B CROSS-VALIDATION')
print('=' * 80)

for pair in V18_LR_PAIRS:
    print(f"\n--- {pair['name']}: {pair['source_lineage']}({pair['ligand']}) → ({pair['receptor']}) ---")

    for target_lin in pair['target_lineages']:
        for tissue in TISSUES:
            # Track A: Check if LIANA found this interaction
            trackA_status = 'NOT_RUN'
            trackA_rank = None

            for disease in ['NL', 'IT']:
                key = f'{tissue}_{disease}'
                if key in liana_pooled:
                    df = liana_pooled[key]
                    match = df[
                        df['ligand_complex'].str.contains(pair['ligand'], case=False, na=False) &
                        df['receptor_complex'].str.contains(pair['receptor'], case=False, na=False) &
                        (df['source'] == pair['source_lineage']) &
                        (df['target'] == target_lin)
                    ]
                    if len(match) > 0:
                        rank = match['magnitude_rank'].min()
                        trackA_status = f'{disease}: rank={rank:.3f}'
                        trackA_rank = rank

            # Track B: Check donor-level test
            trackB = lr_test_df[
                (lr_test_df['tissue'] == tissue) &
                (lr_test_df['source'] == pair['source_lineage']) &
                (lr_test_df['target'] == target_lin) &
                (lr_test_df['ligand'] == pair['ligand']) &
                (lr_test_df['receptor'] == pair['receptor']) &
                (lr_test_df['comparison'] == 'NL→IT')
            ]

            if len(trackB) > 0:
                tb = trackB.iloc[0]
                trackB_status = f"p={tb['p_value']:.4f} {tb['pct_change']:+.1f}%"
                trackB_sig = tb['significant']
            else:
                trackB_status = 'NO DATA'
                trackB_sig = False

            # Verdict
            if trackA_rank is not None and trackA_rank < 0.2 and trackB_sig:
                verdict = '✅ ROBUST (both tracks)'
            elif trackB_sig:
                verdict = '⚠️ Donor-validated only'
            elif trackA_rank is not None and trackA_rank < 0.2:
                verdict = '⚠️ Pooled-only (pseudo-replication risk)'
            else:
                verdict = '❌ Not confirmed'

            print(f'  {tissue} → {target_lin}: TrackA=[{trackA_status}] '
                  f'TrackB=[{trackB_status}] → {verdict}')

  TRACK A vs TRACK B CROSS-VALIDATION

--- TGFB1_TGFBR2: Myeloid(TGFB1) → (TGFBR2) ---
  Liver → CD4_T: TrackA=[NOT_RUN] TrackB=[p=0.3939 +54.5%] → ❌ Not confirmed
  Blood → CD4_T: TrackA=[NOT_RUN] TrackB=[p=0.0159 +264.8%] → ⚠️ Donor-validated only
  Liver → CD8_T: TrackA=[NOT_RUN] TrackB=[p=0.1797 +57.5%] → ❌ Not confirmed
  Blood → CD8_T: TrackA=[NOT_RUN] TrackB=[p=0.0159 +319.3%] → ⚠️ Donor-validated only
  Liver → NK: TrackA=[NOT_RUN] TrackB=[p=0.4848 +40.1%] → ❌ Not confirmed
  Blood → NK: TrackA=[IT: rank=0.396] TrackB=[p=0.0159 +300.5%] → ⚠️ Donor-validated only
  Liver → B: TrackA=[IT: rank=0.876] TrackB=[p=0.2002 +121.4%] → ❌ Not confirmed
  Blood → B: TrackA=[NOT_RUN] TrackB=[p=0.0159 +270.8%] → ⚠️ Donor-validated only

--- TGFB1_TGFBR1: Myeloid(TGFB1) → (TGFBR1) ---
  Liver → CD4_T: TrackA=[NOT_RUN] TrackB=[p=0.3939 +54.4%] → ❌ Not confirmed
  Blood → CD4_T: TrackA=[NOT_RUN] TrackB=[p=0.0159 +272.1%] → ⚠️ Donor-validated only
  Liver → CD8_T: TrackA=[NOT_RUN] TrackB=[p=0.69

---
## Cell 8: Liver vs Blood Interaction Landscape (Track A Discovery)

In [10]:
# ============================================================
# Cell 8: Liver vs Blood top interaction overlap
# ⚠️ Track A (pooled) — interpretation as discovery only
# ============================================================

print('=' * 80)
print('  LIVER vs BLOOD: INTERACTION LANDSCAPE (Track A Discovery)')
print('=' * 80)

for disease in ['NL', 'IT', 'IA']:
    liver_key = f'Liver_{disease}'
    blood_key = f'Blood_{disease}'

    if liver_key not in liana_pooled or blood_key not in liana_pooled:
        continue

    liver_df = liana_pooled[liver_key]
    blood_df = liana_pooled[blood_key]

    def make_key(df):
        return set(df.nsmallest(50, 'magnitude_rank').apply(
            lambda r: f"{r['source']}|{r['ligand_complex']}|{r['receptor_complex']}|{r['target']}",
            axis=1))

    liver_top = make_key(liver_df)
    blood_top = make_key(blood_df)
    overlap = liver_top & blood_top

    pct_overlap = len(overlap) / max(len(liver_top | blood_top), 1) * 100

    print(f'\n{disease}: Liver top50={len(liver_top)}, Blood top50={len(blood_top)}, '
          f'Overlap={len(overlap)} ({pct_overlap:.0f}%)')

    if pct_overlap < 50:
        print(f'  → ✅ Tissue separation IS critical for interactions ({pct_overlap:.0f}% < 50%)')
    else:
        print(f'  → ⚠️ Substantial overlap — tissue separation less critical for interactions')

  LIVER vs BLOOD: INTERACTION LANDSCAPE (Track A Discovery)

NL: Liver top50=50, Blood top50=50, Overlap=32 (47%)
  → ✅ Tissue separation IS critical for interactions (47% < 50%)

IT: Liver top50=50, Blood top50=50, Overlap=29 (41%)
  → ✅ Tissue separation IS critical for interactions (41% < 50%)

IA: Liver top50=50, Blood top50=50, Overlap=32 (47%)
  → ✅ Tissue separation IS critical for interactions (47% < 50%)


---
## Cell 9: Summary Report

In [11]:
# ============================================================
# Cell 9: Final summary
# ============================================================

print('=' * 80)
print('  C11 LIANA — FINAL REPORT')
print('  Simpson\'s Paradox-Aware: 2-Track Design')
print('=' * 80)

print('\n[1] TRACK A (Pooled LIANA — Discovery)')
print(f'  Conditions analyzed: {len(liana_pooled)}')
for key in sorted(liana_pooled.keys()):
    print(f'    {key}: {len(liana_pooled[key])} interactions')

print('\n[2] TRACK B (Donor-Level — Validation)')
for comp in lr_test_df['comparison'].unique():
    comp_df = lr_test_df[lr_test_df['comparison'] == comp]
    sig = comp_df[comp_df['significant']]
    print(f'  {comp}: {len(sig)}/{len(comp_df)} donor-validated interactions')

print('\n[3] V18 HYPOTHESIS VERIFICATION (Track A + Track B)')
for pair in V18_LR_PAIRS:
    name = pair['name']
    # Check Track B NL→IT results
    tb = lr_test_df[
        (lr_test_df['pair_name'] == name) &
        (lr_test_df['comparison'] == 'NL→IT') &
        (lr_test_df['significant'])
    ]
    status = f'✅ {len(tb)} tissue×target validated' if len(tb) > 0 else '❌ Not donor-validated'
    print(f'  {name}: {status}')

print('\n[4] KEY CONCLUSIONS')
print('  - Track A (pooled) results include pseudo-replication')
print('  - Only Track B (donor-level) results are statistically valid')
print('  - Interactions confirmed by BOTH tracks = highest confidence')
print('  - Liver vs Blood overlap % indicates whether tissue separation')
print('    matters for interactions (not just gene expression)')

print('\n[5] FILES')
for f in sorted(os.listdir(RESULTS_DIR)):
    if f.startswith('C11_'):
        size = os.path.getsize(os.path.join(RESULTS_DIR, f))
        print(f'  {f} ({size/1024:.1f} KB)')

print('\n==> C11 Complete.')

  C11 LIANA — FINAL REPORT
  Simpson's Paradox-Aware: 2-Track Design

[1] TRACK A (Pooled LIANA — Discovery)
  Conditions analyzed: 10
    Blood_AR: 4298 interactions
    Blood_CR: 4253 interactions
    Blood_IA: 4873 interactions
    Blood_IT: 4405 interactions
    Blood_NL: 3663 interactions
    Liver_AR: 4756 interactions
    Liver_CR: 5100 interactions
    Liver_IA: 4236 interactions
    Liver_IT: 6111 interactions
    Liver_NL: 4018 interactions

[2] TRACK B (Donor-Level — Validation)
  NL→IT: 11/32 donor-validated interactions
  NL→IA: 14/32 donor-validated interactions
  IT→IA: 0/32 donor-validated interactions
  NL→CR: 0/32 donor-validated interactions
  IA→AR: 0/32 donor-validated interactions

[3] V18 HYPOTHESIS VERIFICATION (Track A + Track B)
  TGFB1_TGFBR2: ✅ 4 tissue×target validated
  TGFB1_TGFBR1: ✅ 3 tissue×target validated
  LGALS9_HAVCR2: ❌ Not donor-validated
  HLA-DRA_CD4: ✅ 2 tissue×target validated
  LGALS9_CD44: ✅ 2 tissue×target validated
  IL2_IL2RA: ❌ Not don